# 六阶段逐层 Representation / Probe / 因子检验

本 Notebook 只负责配置、执行、验证和展示。所有数据处理、模型拟合、统计检验和写盘函数均位于 `src/layer_probe_*.py`。

默认只运行 train/validation 研究流程；最终 test 单元格位于文末且默认关闭。

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs' / 'layer_probe.yaml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'configs' / 'layer_probe.yaml').is_file(), '请从仓库根目录或notebooks目录启动Notebook'
os.chdir(ROOT)

from src.layer_probe_representations import run_representation_stage, validate_representation_artifacts
from src.layer_probe_models import run_sentiment_probe_stage, run_return_probe_stage
from src.layer_probe_panel import run_stock_day_panel_stage, validate_stock_day_artifacts
from src.layer_probe_factors import run_factor_validation_stage
from src.layer_probe_pipeline import (
    assert_preflight, load_layer_probe_config, plot_factor_summary,
    plot_logit_comparison, plot_return_curve, plot_sentiment_curve,
    preflight_report, validate_factor_outputs, validate_pipeline_outputs,
    validate_return_probe_outputs, validate_sentiment_probe_outputs,
    evaluation_output_directory,
)

CONFIG_PATH = ROOT / 'configs' / 'layer_probe.yaml'
config = load_layer_probe_config(CONFIG_PATH)
pd.set_option('display.max_columns', 100)
print('repo:', ROOT)
print('config:', CONFIG_PATH)

## 0. 执行开关

阶段1很昂贵但可复用。已存在且验收通过时，可把对应开关改为 `False`。开发阶段不得打开最终 test。

In [ ]:
RUN_STAGE_1_REPRESENTATIONS = True
RUN_STAGE_2_SENTIMENT = True
RUN_STAGE_3_STOCK_DAY = True
RUN_STAGE_4_RETURN_PROBE = True
RUN_STAGE_5_6_FACTOR_VALIDATION = True
RUN_FINAL_TEST_CELL = False

assert config['sentiment_probe']['open_final_test'] is False, '开发阶段必须关闭情绪test'
assert config['return_probe']['open_final_test'] is False, '开发阶段必须关闭收益test'
assert config['strict_test']['open_final_test'] is False, '开发阶段必须关闭因子test'

## 1. 全局预检

在任何GPU前向之前，确认文本、checkpoint、tokenizer、收益文件和三段时间窗口。行业/规模暴露缺失只影响分组检验，不阻断主流程。

In [ ]:
preflight = preflight_report(config)
display(preflight)
assert_preflight(preflight)

## 阶段1：一次forward提取Layer 0～12 CLS

验收包括：checkpoint/tokenizer哈希固定、模型全部参数无梯度、dropout关闭、13层齐全、metadata/head/NPY逐行对齐。

In [ ]:
if RUN_STAGE_1_REPRESENTATIONS:
    representation_artifacts = run_representation_stage(config)
representation_check = validate_representation_artifacts(config['output']['representations'])
display(pd.Series(representation_check, name='stage_1_validation'))
representation_manifest = json.loads((Path(config['output']['representations']) / 'representation_manifest.json').read_text(encoding='utf-8'))
display(pd.Series(representation_manifest['model_validation']))
assert representation_manifest['shape'][1] == 13
assert representation_manifest['model_validation']['trainable_parameter_count'] == 0
assert representation_manifest['model_validation']['dropout_modules_in_training_mode'] == []

## 阶段2：13个情绪 Logistic Probe

每层标准化和Logistic只在train拟合，C只由validation AUC选择。原fc头在同一批OOS样本上比较。若标签映射尚未确认，只能把结果称为class-1诊断。

In [ ]:
if RUN_STAGE_2_SENTIMENT:
    sentiment_output = run_sentiment_probe_stage(config)
sentiment_check = validate_sentiment_probe_outputs(config['output']['sentiment_probe'])
display(pd.Series(sentiment_check, name='stage_2_validation'))
sentiment_run_dir = evaluation_output_directory(config['output']['sentiment_probe'])
sentiment_metrics = pd.read_csv(sentiment_run_dir / 'sentiment_metrics.csv')
compression = pd.read_csv(sentiment_run_dir / 'probability_compression_diagnostic.csv')
display(sentiment_metrics[sentiment_metrics['split'].eq('validation')].sort_values(['model_kind', 'layer']))
display(compression)
plot_sentiment_curve(config['output']['sentiment_probe'], split='validation');
plot_logit_comparison(config['output']['sentiment_probe'], split='validation');

## 阶段3：股票日13层representation panel

按 `symbol × trading_date` 对各层CLS取均值，生成 `n_texts`，连接未来1/5/20日行业调整收益。用20日Label结束日清除跨越下一分区的样本。

In [ ]:
if RUN_STAGE_3_STOCK_DAY:
    stock_day_output = run_stock_day_panel_stage(config)
stock_day_check = validate_stock_day_artifacts(config['output']['stock_day_panel'])
display(pd.Series(stock_day_check, name='stage_3_validation'))
stock_day_panel = pd.read_parquet(Path(config['output']['stock_day_panel']) / 'stock_day_panel.parquet')
display(stock_day_panel.groupby('split').agg(rows=('representation_row', 'size'), dates=('trading_date', 'nunique'), symbols=('symbol', 'nunique'), mean_texts=('n_texts', 'mean')))
display(pd.read_csv(Path(config['output']['stock_day_panel']) / 'stock_day_audit.csv'))
assert not stock_day_panel.duplicated(['symbol', 'trading_date']).any()

## 阶段4：逐层Ridge收益Probe（validation开发）

主目标为5日行业调整收益的日内截面排名。alpha只根据validation日均Rank IC选择；当前单元格不会生成test指标。

In [ ]:
if RUN_STAGE_4_RETURN_PROBE:
    return_output = run_return_probe_stage(config)
return_check = validate_return_probe_outputs(config['output']['return_probe'])
display(pd.Series(return_check, name='stage_4_validation'))
return_run_dir = evaluation_output_directory(config['output']['return_probe'])
return_metrics = pd.read_csv(return_run_dir / 'return_probe_metrics.csv')
display(return_metrics.sort_values(['split', 'layer']))
assert set(return_metrics['split']) == {'validation'}
plot_return_curve(config['output']['return_probe'], split='validation');

## 阶段5～6：跨层因子与严格validation检验

候选包括13个单层、层间共识、分歧、deep-minus-middle、deep residual和PCA层因子。deep residual与PCA只在历史reference拟合。所有候选统一执行BH多重检验修正。

In [ ]:
if RUN_STAGE_5_6_FACTOR_VALIDATION:
    factor_output = run_factor_validation_stage(config)
factor_check = validate_factor_outputs(config['output']['factor_validation'])
display(pd.Series(factor_check, name='stage_5_6_validation'))
factor_run_dir = evaluation_output_directory(config['output']['factor_validation'])
factor_summary = pd.read_csv(factor_run_dir / 'factor_summary.csv').sort_values('rank_ic', ascending=False)
monotonicity = pd.read_csv(factor_run_dir / 'quantile_monotonicity.csv').sort_values('top_minus_bottom_mean', ascending=False)
incremental = pd.read_csv(factor_run_dir / 'incremental_ic.csv').sort_values('rank_ic', ascending=False)
display(factor_summary)
display(monotonicity)
display(incremental)
plot_factor_summary(config['output']['factor_validation']);

## Validation后预注册最终因子

根据上面的validation结果，把最终保留的因子名称写入 `configs/layer_probe.yaml -> strict_test.selected_factors`。在此之前不要打开test。建议同时保存研究说明或Git提交，固定选择。

In [ ]:
candidate_names = pd.read_parquet(evaluation_output_directory(config['output']['factor_validation']) / 'candidate_factor_matrix.parquet').columns
candidate_names = [name for name in candidate_names if name.startswith('single_layer_') or name in {'layer_consensus', 'layer_disagreement', 'deep_minus_middle', 'deep_residual'} or name.startswith('layer_pca_')]
display(pd.Series(candidate_names, name='可预注册因子名'))

## 最终测试集：只运行一次

执行前必须同时完成：

1. `return_probe.open_final_test: true`；
2. `strict_test.open_final_test: true`；
3. `strict_test.selected_factors` 非空且已预注册；
4. 将下方 `RUN_FINAL_TEST_CELL` 改为 `True`。

成功执行后会写入不可重复开启的marker；再次运行将直接报错。

In [ ]:
if RUN_FINAL_TEST_CELL:
    final_config = load_layer_probe_config(CONFIG_PATH)
    assert final_config['return_probe']['open_final_test'] is True
    assert final_config['strict_test']['open_final_test'] is True
    assert len(final_config['strict_test']['selected_factors']) > 0
    final_preflight = preflight_report(final_config)
    display(final_preflight)
    assert_preflight(final_preflight)
    run_return_probe_stage(final_config)
    run_factor_validation_stage(final_config)
    final_return_dir = evaluation_output_directory(final_config['output']['return_probe'], final_test=True)
    final_factor_dir = evaluation_output_directory(final_config['output']['factor_validation'], final_test=True)
    display(pd.Series(validate_return_probe_outputs(final_return_dir), name='final_return_validation'))
    display(pd.Series(validate_factor_outputs(final_factor_dir), name='final_factor_validation'))
    display(pd.read_csv(final_return_dir / 'return_probe_metrics.csv').query("split == 'test'"))
    display(pd.read_csv(final_factor_dir / 'factor_summary.csv'))
else:
    print('最终test保持关闭。')

## 全管线最终验收

In [ ]:
pipeline_validation = validate_pipeline_outputs(config)
display(pipeline_validation)
assert not pipeline_validation['status'].eq('failed').any()